# Bookended nonresonant lab XRR fit

Transplant the resonant (283.7 eV) graded book-ended ZnPc profile onto Cu-K\(\alpha\) lab XRR and refit against the data. Free film parameters for now: `total_thick`, interface densities (`density_vac`, `density_si`), Oxide thick/rough, and s-channel instrument terms.

Run from the **refl-analysis** project root so `utils` resolves. Saves a refloxide `Objective` pickle under `@models/xrr/znpc/nonres/` for reuse (same pattern as `real_data_fit.ipynb` and `graded/graded_fit.pkl`).

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from refnx.analysis import CurveFitter, Transform

from refloxide.data import ReflectDataset
from refloxide.model import BookendedComponent, MaterialSLD, ReflectModel
from refloxide.objective import Objective
from refloxide.pxr.energy.bookended import BookendedOrientationProfile
from refloxide.pxr.energy.ooc import OocAnchor

from utils import models_root, notebooks_fitting
from utils.helpers.plotting_helper import set_plotting_defaults
from utils.models import configure_refloxide_fitting

configure_refloxide_fitting()
set_plotting_defaults()

ENERGY_EV = 8.04e3
_HC_EV_ANGSTROM = 12398.42
WAVELENGTH_A = _HC_EV_ANGSTROM / ENERGY_EV

print(f"probe energy: {ENERGY_EV:.2f} eV ({ENERGY_EV / 1e3:.3f} keV)")
print(f"wavelength: {WAVELENGTH_A:.4f} A (Cu K-alpha lab source)")

## Paths

Resonant graded anchors come from `graded_fit_summary.json` (extract once via `scripts/extract_graded_bookended_fit.py`). Lab CSV lives alongside the other fitting notebooks.

In [ ]:
GRADED_SUMMARY_PATH = models_root / "xrr/znpc/graded/graded_fit_summary.json"
CSV_PATH = notebooks_fitting / "HiRes_BigRegion_ExcelMerge_CSV.csv"
NONRES_DIR = models_root / "xrr/znpc/nonres"
FIT_OUT_PATH = NONRES_DIR / "nonres_graded_fit.pkl"
SUMMARY_OUT_PATH = NONRES_DIR / "nonres_graded_fit_summary.json"

for path in (GRADED_SUMMARY_PATH, CSV_PATH):
    if not path.exists():
        raise FileNotFoundError(path)

summary = json.loads(GRADED_SUMMARY_PATH.read_text())
ooc_csv = Path(summary["ooc_csv"])
if not ooc_csv.exists():
    raise FileNotFoundError(ooc_csv)
ZNPC_OOC = OocAnchor.from_file(ooc_csv)

RESONANT_ENERGY_EV = float(summary["energy"])
film_params = summary["film"]["params"]

print(f"resonant reference fit: {RESONANT_ENERGY_EV:.1f} eV")
print(f"nonresonant probe fit:  {ENERGY_EV:.2f} eV")
for name, value in film_params.items():
    print(f"  {name}: {value:.4f}")

## Load lab XRR CSV

In [ ]:
frame = pl.read_csv(CSV_PATH)
angle_deg = frame["IncidentAngle(deg)"].to_numpy()
intensity = frame["Intensity(a.u.)"].to_numpy()
sigma_i = frame["Sigma_I(a.u.)"].to_numpy()

q = (4.0 * np.pi / WAVELENGTH_A) * np.sin(np.radians(angle_deg))
norm = float(intensity.max())
r = intensity / norm
r_err = sigma_i / norm
print(f"q range = [{q.min():.4f}, {q.max():.4f}] 1/A, {len(q)} points")

dataset = ReflectDataset(
    q=q,
    energy=np.full_like(q, ENERGY_EV),
    pol=np.full(q.shape, "s", dtype=object),
    r=r,
    r_err=r_err,
)

## Model setup

In [ ]:
BKG_ESTIMATE = 0.0
layers = {layer["name"]: layer for layer in summary["layers"]}


def build_slab(
    name: str,
    *,
    vary_rough: bool = False,
    rough_bounds: tuple[float, float] = (0.0, 30.0),
):
    layer = layers[name]
    sld = MaterialSLD(layer["formula"], density=layer["density"], name=name)
    slab = sld(layer["thick"], layer["rough"])
    slab.thick.vary = False
    sld.density.vary = False
    if vary_rough:
        slab.rough.setp(vary=True, bounds=rough_bounds)
    else:
        slab.rough.vary = False
    return slab


def freeze_instrumentation(model: ReflectModel) -> None:
    model.scale_s.at(ENERGY_EV).setp(vary=True, bounds=(0.8, 1.2))
    model.theta_offset_s.at(ENERGY_EV).setp(vary=True, bounds=(-0.8, 0.8))
    model.bkg.at(ENERGY_EV).value = BKG_ESTIMATE
    model.bkg.at(ENERGY_EV).vary = False
    model.scale_p.at(ENERGY_EV).vary = False
    model.theta_offset_p.at(ENERGY_EV).vary = False


_oxide_thick = layers["Oxide"]["thick"]
_oxide_rough_ceiling = float(np.sqrt(2.0 * np.pi) * _oxide_thick / 2.0)

BOUNDS = {
    "surface_roughness": (3.0, 20.0),
    "oxide_rough": (0.0, _oxide_rough_ceiling),
    "density_vac": (1.5, 2.5),
    "density_si": (0.0, 1.3),
}

REFERENCE = {
    "surface_thick": 7.89151,
    "surface_rough": 6.28961,
    "bulk_thick": 125.596,
    "interface_thick": 18.2626,
    "oxide_thick": 9.84672,
    "oxide_rho": 2.0692,
    "substrate_rough": 0.5,
    "substrate_rho": 2.21255,
}

graded_film = BookendedOrientationProfile(
    ooc=ZNPC_OOC,
    energy=summary["energy"],
    num_slabs=summary["film"]["num_slabs"],
    mesh_constant=summary["film"]["mesh_constant"],
    name="ZnPc",
    **film_params,
)
graded_structure = (
    build_slab("Vacuum")
    | BookendedComponent(graded_film)
    | build_slab("Oxide", vary_rough=True, rough_bounds=BOUNDS["oxide_rough"])
    | build_slab("Substrate")
)

graded_model = ReflectModel(graded_structure, parallel=False)
graded_film.total_thick.setp(value=150, vary=True, bounds=(100, 180))
graded_film.surface_roughness.setp(
    value=REFERENCE["surface_rough"], vary=False, bounds=BOUNDS["surface_roughness"]
)
graded_film.tau_vac.setp(value=REFERENCE["surface_thick"], vary=False)
graded_film.tau_si.setp(value=REFERENCE["interface_thick"], vary=False)
graded_film.density_vac.setp(vary=True, bounds=BOUNDS["density_vac"])
graded_film.density_si.setp(vary=True, bounds=BOUNDS["density_si"])
graded_film.density_bulk.vary = False

oxide_slab = graded_structure.slab("Oxide")
oxide_slab.thick.setp(value=REFERENCE["oxide_thick"], vary=True, bounds=(5, 12.0))
oxide_slab.rough.setp(vary=True, bounds=(0, 12))
oxide_slab.sld.density.setp(value=REFERENCE["oxide_rho"], vary=False)

substrate_slab = graded_structure.slab("Substrate")
substrate_slab.rough.value = REFERENCE["substrate_rough"]
substrate_slab.sld.density.value = REFERENCE["substrate_rho"]

freeze_instrumentation(graded_model)

graded_objective = Objective(
    graded_model, dataset, transform=Transform("logY"), nc_constraint=True
)
print(f"free parameters: {len(graded_objective.varying_parameters())}")
print(f"logl before fit: {graded_objective.logl():.3f}")
print(graded_objective.varying_parameters())

## Fit

In [ ]:
CurveFitter(graded_objective).fit(
    method="differential_evolution", popsize=30, polish=True, seed=1
)
print(f"logl after fit:  {graded_objective.logl():.3f}")
print(f"chisqr after fit: {graded_objective.chisqr():.3f}")
print(f"recovered total_thick   = {graded_film.total_thick.value:.2f} A")
print(f"recovered density_vac   = {graded_film.density_vac.value:.4f} g/cm^3")
print(f"recovered density_si    = {graded_film.density_si.value:.4f} g/cm^3")
print(f"recovered density_bulk  = {graded_film.density_bulk.value:.4f} g/cm^3 (fixed)")
print(f"recovered oxide thick   = {oxide_slab.thick.value:.2f} A")
print(f"recovered oxide rough   = {oxide_slab.rough.value:.2f} A")
print(f"recovered scale_s       = {graded_model.scale_s.at(ENERGY_EV).value:.4f}")
print(
    f"recovered theta_offset_s = "
    f"{graded_model.theta_offset_s.at(ENERGY_EV).value:.4f}"
)
print(graded_objective.varying_parameters())

## Save fitted objective

Writes `@models/xrr/znpc/nonres/nonres_graded_fit.pkl` plus a portable JSON summary. Reload the pickle in downstream notebooks without rerunning DE.

In [ ]:
NONRES_DIR.mkdir(parents=True, exist_ok=True)

with FIT_OUT_PATH.open("wb") as f:
    pickle.dump(graded_objective, f, protocol=pickle.HIGHEST_PROTOCOL)

film_param_names = (
    "total_thick",
    "surface_roughness",
    "tau_si",
    "tau_vac",
    "alpha_bulk",
    "alpha_si",
    "alpha_vac",
    "density_bulk",
    "density_si",
    "density_vac",
)
fit_summary = {
    "source_resonant_summary": str(GRADED_SUMMARY_PATH),
    "resonant_energy_ev": RESONANT_ENERGY_EV,
    "energy_ev": ENERGY_EV,
    "wavelength_a": WAVELENGTH_A,
    "dataset_csv": str(CSV_PATH),
    "ooc_csv": str(ooc_csv),
    "film": {
        "num_slabs": int(graded_film.num_slabs),
        "mesh_constant": float(graded_film.mesh_constant),
        "params": {
            name: float(getattr(graded_film, name).value or 0.0)
            for name in film_param_names
        },
    },
    "layers": [
        {
            "name": slab.name.rsplit("_", 1)[0],
            "thick": float(slab.thick.value or 0.0),
            "rough": float(slab.rough.value or 0.0),
            "density": float(slab.sld.density.value or 0.0),
            "formula": getattr(slab.sld, "formula", ""),
        }
        for slab in graded_model.structure.components
        if hasattr(slab, "thick")
    ],
    "instrumentation": {
        "scale_s": float(graded_model.scale_s.at(ENERGY_EV).value or 0.0),
        "theta_offset_s": float(graded_model.theta_offset_s.at(ENERGY_EV).value or 0.0),
        "bkg": float(graded_model.bkg.at(ENERGY_EV).value or 0.0),
    },
    "fit": {
        "logl": float(graded_objective.logl()),
        "chisqr": float(graded_objective.chisqr()),
    },
}
SUMMARY_OUT_PATH.write_text(json.dumps(fit_summary, indent=2))
print(f"saved objective to {FIT_OUT_PATH}")
print(f"saved summary to {SUMMARY_OUT_PATH}")

## Supplemental figure (fig_5 colors)

Two panels at PRL width 3.55 in, aspect \(2.55{:}2.9\): reflectivity (top) and Nevot-Croce density with \(\tau\) markers (bottom). Styling follows `manuscript/fig_5_graded.ipynb` (`set_plotting_defaults`, `C_GRADED`, `TAU_COLOR`).

In [ ]:
def nevot_croce_density(film, *, n_points: int = 2000, n_sigma: int = 5):
    """Convolve book-ended density with the vacuum-side Nevot-Croce roughness."""
    total = float(film.total_thick.value or 0.0)
    sigma = float(film.surface_roughness.value or 0.0)
    depth = np.linspace(0.0, total, n_points)
    rho_sharp = np.asarray(film.local_density(depth), dtype=float)
    if sigma <= 0.0:
        return depth, rho_sharp, rho_sharp.copy()

    dz = depth[1] - depth[0]
    pad = int(np.ceil(n_sigma * sigma / dz))
    z_ext = np.concatenate(
        [depth[0] + dz * np.arange(-pad, 0), depth, depth[-1] + dz * np.arange(1, pad + 1)]
    )
    rho_ext = np.empty_like(z_ext)
    below, above = z_ext < 0.0, z_ext > total
    inside = ~(below | above)
    rho_ext[below] = 0.0
    rho_ext[inside] = np.asarray(film.local_density(z_ext[inside]), dtype=float)
    rho_ext[above] = float(film.local_density(total))

    kernel = np.exp(-0.5 * (dz * np.arange(-pad, pad + 1) / sigma) ** 2)
    kernel /= kernel.sum()
    rho_rough = np.convolve(rho_ext, kernel, mode="same")[pad : pad + n_points]
    return depth, rho_sharp, rho_rough


_TAB20 = plt.colormaps["tab20"]
C_S = _TAB20(2)
C_GRADED = _TAB20(2)
TAU_COLOR = "#0b7f8a"
TAU_LW = 0.85
TAU_LS = (0, (4, 3))
RHO_YLIM = (0.5, 2.2)

film = graded_film
TOTAL = float(film.total_thick.value or 0.0)
TAU_VAC = float(film.tau_vac.value or 0.0)
TAU_SI = float(film.tau_si.value or 0.0)
Z_G, RHO_SHARP, RHO_ROUGH = nevot_croce_density(film)

PRL_WIDTH_IN = 3.55
FIG_HEIGHT_IN = PRL_WIDTH_IN * (2.9 / 2.55)

q_fine = np.linspace(q.min(), q.max(), 1500)
r_fine = graded_model(q_fine, ENERGY_EV).s

print(f"figure probe energy: {ENERGY_EV:.2f} eV ({ENERGY_EV / 1e3:.3f} keV)")


def highlight_interface(ax, *, halo=True, labels=True):
    for z_tau, txt, ha, dx in (
        (TAU_VAC, r"$\tau_\mathrm{vac}$", "left", 3),
        (TOTAL - TAU_SI, r"$\tau_\mathrm{Si}$", "right", -3),
    ):
        if halo:
            ax.axvline(
                z_tau,
                color="white",
                lw=TAU_LW + 1.1,
                solid_capstyle="butt",
                zorder=4,
            )
        ax.axvline(
            z_tau,
            color=TAU_COLOR,
            lw=TAU_LW,
            ls=TAU_LS,
            solid_capstyle="butt",
            zorder=5,
        )
        if labels:
            ax.annotate(
                txt,
                xy=(z_tau, 0.5),
                xycoords=("data", "axes fraction"),
                xytext=(dx, 0),
                textcoords="offset points",
                va="top",
                ha=ha,
                color=TAU_COLOR,
                fontsize=7,
            )


fig, (ax_fit, ax_rho) = plt.subplots(
    2,
    1,
    figsize=(PRL_WIDTH_IN, FIG_HEIGHT_IN),
    dpi=400,
    layout="constrained",
    gridspec_kw={"height_ratios": (1.2, 1.0), "hspace": 0.08},
)

ax_fit.errorbar(
    q,
    r,
    yerr=r_err,
    fmt="o",
    lw=0,
    ms=2.8,
    elinewidth=0.45,
    capsize=1.2,
    capthick=0.45,
    color=C_S,
    ecolor=C_S,
    label="data",
    zorder=10,
)
ax_fit.plot(q_fine, r_fine, color=C_GRADED, lw=1.1, label="graded fit", zorder=5)
ax_fit.set_yscale("log")
ax_fit.set_xlabel(r"$q$ ($\mathrm{\AA}^{-1}$)")
ax_fit.set_ylabel("Reflectivity")
ax_fit.legend(frameon=False, handlelength=1.5, loc="upper right")
ax_fit.annotate(
    "(a)",
    xy=(-0.18, 1.02),
    xycoords="axes fraction",
    fontsize=9,
    ha="left",
    va="bottom",
    fontweight="bold",
)

surf = Z_G <= TAU_VAC
ax_rho.plot(
    Z_G[surf],
    RHO_SHARP[surf],
    color=C_GRADED,
    ls=":",
    lw=0.8,
    zorder=4,
)
ax_rho.plot(Z_G, RHO_ROUGH, color=C_GRADED, lw=1.0, zorder=5, label="graded")
highlight_interface(ax_rho, halo=True, labels=True)
ax_rho.set_xlim(0.0, TOTAL)
ax_rho.set_ylim(*RHO_YLIM)
ax_rho.set_xlabel(r"depth $z$ ($\mathrm{\AA}$)")
ax_rho.set_ylabel(r"density (g/cm$^{3}$)")
ax_rho.legend(frameon=False, handlelength=1.5, loc="lower center")
ax_rho.annotate(
    "(b)",
    xy=(-0.18, 1.02),
    xycoords="axes fraction",
    fontsize=9,
    ha="left",
    va="bottom",
    fontweight="bold",
)

fig.align_ylabels()
out_png = notebooks_fitting / "nonres_graded_fit_density.png"
fig.savefig(out_png, dpi=400)
plt.show()
print(f"saved {out_png}")
print(f"tau_vac = {TAU_VAC:.2f} A, tau_Si edge = {TOTAL - TAU_SI:.2f} A")